# TFM — 03. Feature Engineering: Accidentes de tráfico en Madrid (2012-2018)
**Autora:** Meritxell Abellan Collado

## Contexto y objetivo de este notebook

En el notebook anterior (`02_Preprocessing.ipynb`) se dejó el dataset limpio a nivel accidente. Antes de construir features, se resolvió una pregunta que condiciona todo lo que sigue: **¿qué información está realmente disponible en el momento en que se necesita la predicción?**

**Pregunta central del TFM:** *¿Es posible predecir, con la información disponible en el momento del aviso de un accidente de tráfico en Madrid (ubicación, momento, condiciones meteorológicas y tipo de siniestro), la probabilidad de que resulte grave, con el fin de apoyar la priorización de recursos de emergencia (SAMUR / Policía Municipal / Emergencias 112)?*

Esto obliga a distinguir, para cada variable, si se conoce **en el momento del aviso** o solo **después**, cuando ya han llegado los servicios de emergencia y se ha identificado a las personas implicadas. De ahí surgen **tres bloques de trabajo** en este notebook, que no compiten entre sí sino que responden a preguntas distintas:

| Bloque | Variables que usa | Para qué sirve |
|---|---|---|
| **1. Modelo operativo** | Solo las disponibles en el aviso (lugar, momento, clima, tipo de accidente/vehículo) + agregados históricos | Es el modelo real: el que se podría usar en producción para apoyar el despacho de recursos |
| **2. Modelo de referencia** | Todo lo anterior + variables de persona (edad de riesgo, nº de implicados, nº de víctimas) | Cuantifica cuánto se "pierde" por no saber aún quién está implicado — no es desplegable, es un techo de referencia |
| **3. Análisis explicativo** | Variables de persona/vehículo tal cual | No es para predecir, es para identificar factores de riesgo y orientar campañas de seguridad vial (se retoma en el notebook de modelización/interpretabilidad) |

Este notebook construye las features y dos matrices de diseño (operativo y referencia). El bloque 3 se apoya en las mismas variables ya calculadas, y se explota con más detalle en el notebook de modelización (SHAP / importancia de variables).


## 0. Carga de datos

In [1]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)

train = pd.read_csv('../data/processed/train_temporal.csv')
test = pd.read_csv('../data/processed/test_temporal.csv')

print(f'Train: {train.shape[0]:,} filas | Test: {test.shape[0]:,} filas')
print(f'Tasa de gravedad train: {train["GRAVE"].mean()*100:.2f}% | test: {test["GRAVE"].mean()*100:.2f}%')

Train: 58,095 filas | Test: 10,678 filas
Tasa de gravedad train: 9.86% | test: 8.19%

**Por qué se parte de `train_temporal` / `test_temporal` y no del dataset completo:** cualquier estadístico que se calcule a partir de aquí (medias históricas, categorías de referencia, etc.) tiene que aprenderse **solo con `train`** y aplicarse después a `test`, exactamente igual que se haría con datos nuevos en producción. Trabajar ya con los dos conjuntos separados desde la primera celda evita el error más habitual en este tipo de proyecto: calcular una estadística con todo el dataset y filtrar sin querer información del futuro hacia el pasado.

## 1. Recuperar señal de las variables de persona sin romper la disponibilidad temporal

Aunque no se pueda saber, en el aviso de *este* accidente, si habrá un peatón implicado, sí se puede saber el **perfil histórico** de esa zona: ¿qué proporción de accidentes pasados en ese distrito implicaban un peatón o una moto? Esto es información legítima en el momento del aviso, porque no describe el accidente actual — describe el patrón histórico del contexto (el mismo tipo de dato que usaría un sistema real de despacho de emergencias con acceso a series históricas de siniestralidad).

**Por qué se usa suavizado (`m=50`) y no la media simple por grupo:** un distrito con pocos accidentes podría tener, por puro azar, una tasa de gravedad histórica extrema (0% o 100%) que no generalizaría bien. El suavizado tipo Bayes empírico combina la media del grupo con la media global, ponderando por el tamaño de la muestra: grupos grandes se quedan casi con su propia media; grupos pequeños se acercan a la media global. Es la misma idea que un *target encoding* clásico, pero evitando que categorías con pocos casos introduzcan ruido.

**Por qué se agrega por `DISTRITO`** y no por `TIPO_VIA` o `TIPO ACCIDENTE`: esas dos variables se van a codificar más abajo con *one-hot* directamente, así que un agregado histórico calculado sobre el mismo grupo sería puramente redundante (cada fila del agregado sería una función exacta de las columnas *one-hot*, sin aportar información nueva). `DISTRITO` no se codifica con *one-hot* en este proyecto (ver sección 3), así que aquí sí aporta señal genuina.


In [2]:
def smoothed_target_encode(train_df, col, target, m):
    """Codificación de una categórica por la media (suavizada) de `target` en ese grupo,
    ajustada SOLO con datos de entrenamiento."""
    global_mean = train_df[target].mean()
    agg = train_df.groupby(col)[target].agg(['mean', 'count'])
    smoothed = (agg['count'] * agg['mean'] + m * global_mean) / (agg['count'] + m)
    return smoothed, global_mean

def apply_encoding(df, col, mapping, global_mean, new_col):
    """Aplica el mapeo aprendido en train; categorías no vistas -> media global."""
    df[new_col] = df[col].map(mapping).fillna(global_mean)
    return df

M = 50  # fuerza de suavizado: equivale al peso de ~50 accidentes de la media global

for col, target, nuevo_nombre in [
    ('DISTRITO', 'GRAVE',          'TASA_GRAVEDAD_HIST_DISTRITO'),
    ('DISTRITO', 'INCLUYE_PEATON', 'PCT_HIST_PEATON_DISTRITO'),
    ('DISTRITO', 'INCLUYE_MOTO',   'PCT_HIST_MOTO_DISTRITO'),
]:
    mapping, global_mean = smoothed_target_encode(train, col, target, M)
    train = apply_encoding(train, col, mapping, global_mean, nuevo_nombre)
    test = apply_encoding(test, col, mapping, global_mean, nuevo_nombre)

print('Nulos tras aplicar el mapeo a test (categorías no vistas):',
      test[['TASA_GRAVEDAD_HIST_DISTRITO','PCT_HIST_PEATON_DISTRITO','PCT_HIST_MOTO_DISTRITO']].isna().sum().sum())
train[['DISTRITO','TASA_GRAVEDAD_HIST_DISTRITO','PCT_HIST_PEATON_DISTRITO','PCT_HIST_MOTO_DISTRITO']].drop_duplicates().sort_values('TASA_GRAVEDAD_HIST_DISTRITO').head()

Nulos tras aplicar el mapeo a test (categorías no vistas): 0
           DISTRITO  TASA_GRAVEDAD_HIST_DISTRITO  PCT_HIST_PEATON_DISTRITO  PCT_HIST_MOTO_DISTRITO
54  MONCLOA-ARAVACA                     0.088564                  0.105627                0.378698
5         SALAMANCA                     0.090077                  0.124912                0.534267
6         CHAMARTIN                     0.090957                  0.119427                0.514585
3     CIUDAD LINEAL                     0.092738                  0.144342                0.407117
2            TETUAN                     0.092766                  0.170772                0.526518

No hay categorías de `DISTRITO` en `test` que no existieran ya en `train` (0 nulos tras aplicar el mapeo), pero el `.fillna(global_mean)` queda como salvaguarda: si en el futuro apareciera un distrito nuevo (poco probable, pero el pipeline debe ser robusto), se le asignaría la media global en vez de un nulo que rompería el modelo.

## 2. Variables temporales: codificación cíclica

`HORA` (0-23) y `MES` (1-12) son variables cíclicas: la hora 23 está tan cerca de la 0 como la 22 lo está de la 23, pero un modelo que reciba `HORA` como un número lineal no lo sabe — para él, 0 y 23 están todo lo lejos posible. Lo mismo pasa con diciembre (12) y enero (1). Se codifican con seno/coseno, la forma estándar de representar periodicidad sin imponer un salto artificial en la frontera del ciclo.


In [3]:
for df_ in [train, test]:
    df_['HORA_SIN'] = np.sin(2 * np.pi * df_['HORA'] / 24)
    df_['HORA_COS'] = np.cos(2 * np.pi * df_['HORA'] / 24)
    df_['MES_SIN'] = np.sin(2 * np.pi * df_['MES'] / 12)
    df_['MES_COS'] = np.cos(2 * np.pi * df_['MES'] / 12)

train[['HORA','HORA_SIN','HORA_COS']].drop_duplicates().sort_values('HORA')

     HORA      HORA_SIN      HORA_COS
34      0  0.000000e+00  1.000000e+00
132     1  2.588190e-01  9.659258e-01
0       2  5.000000e-01  8.660254e-01
134     3  7.071068e-01  7.071068e-01
12      4  8.660254e-01  5.000000e-01
3       5  9.659258e-01  2.588190e-01
295     6  1.000000e+00  6.123234e-17
4       7  9.659258e-01 -2.588190e-01
36      8  8.660254e-01 -5.000000e-01
39      9  7.071068e-01 -7.071068e-01
13     10  5.000000e-01 -8.660254e-01
43     11  2.588190e-01 -9.659258e-01
15     12  1.224647e-16 -1.000000e+00
17     13 -2.588190e-01 -9.659258e-01
20     14 -5.000000e-01 -8.660254e-01
96     15 -7.071068e-01 -7.071068e-01
22     16 -8.660254e-01 -5.000000e-01
46     17 -9.659258e-01 -2.588190e-01
6      18 -1.000000e+00 -1.836970e-16
8      19 -9.659258e-01  2.588190e-01
9      20 -8.660254e-01  5.000000e-01
77     21 -7.071068e-01  7.071068e-01
10     22 -5.000000e-01  8.660254e-01
11     23 -2.588190e-01  9.659258e-01

> **Nota:** `ES_FINDE`, ya construida en el preprocesado, se mantiene como variable binaria adicional — no es redundante con `MES_SIN/COS`, describe un patrón semanal, no anual. `AÑO`, `FECHA`, `RANGO HORARIO` y `DIA SEMANA` (las versiones sin procesar) se descartan como features: `AÑO` porque en 2018 (test) el modelo vería un valor que nunca estuvo en el rango de entrenamiento (2012-2017), lo que puede confundir a algunos algoritmos con una variable que en la práctica es solo un identificador temporal, no una causa; y `RANGO HORARIO`/`DIA SEMANA` porque ya están representadas por sus versiones numéricas/cíclicas.

## 3. Codificación de categóricas: una decisión distinta para cada variable, no una regla única

No todas las categóricas se codifican igual — la elección depende de la cardinalidad y de si ya existe una alternativa mejor:

- **`TIPO_VIA`** (9 categorías) y **`TIPO ACCIDENTE`** (11 categorías): cardinalidad baja, sin orden natural entre categorías → **one-hot**. Es la opción más simple e interpretable, y no asume ninguna relación de orden o distancia entre "autovía" y "calle" que no exista en realidad.
- **`DISTRITO`** (21 categorías): ya se ha usado en la sección 1 como base de los agregados históricos suavizados. Añadir además un *one-hot* de 20 columnas sería redundante y aumentaría mucho la dimensionalidad para lo que aporta — se deja representado únicamente a través de esos 3 agregados.

**Por qué se elimina una categoría de referencia en cada one-hot (`drop first`):** si se incluyen las *k* categorías completas de una variable mutuamente excluyente, la última es perfectamente predecible a partir de las demás (si no es autovía, ni avenida, ni calle... y así sucesivamente, tiene que ser la que falta) — esto es la "trampa de la variable ficticia" (*dummy variable trap*), y genera colinealidad perfecta que perjudica a modelos lineales. Se elige como categoría de referencia la más frecuente de cada variable (`CALLE` y `COLISIÓN DOBLE`), no la primera por orden alfabético, para que los coeficientes de un modelo lineal se interpreten como "efecto frente al caso más común", que es más intuitivo que compararlo con una categoría minoritaria arbitraria.


In [4]:
REF_VIA = 'CALLE'            # categoria mas frecuente de TIPO_VIA (52.5% de los accidentes)
REF_ACC = 'COLISIÓN DOBLE'   # categoria mas frecuente de TIPO ACCIDENTE (55.4% de los accidentes)

cats_tipo_via = [c for c in sorted(train['TIPO_VIA'].unique()) if c != REF_VIA]
cats_tipo_acc = [c for c in sorted(train['TIPO ACCIDENTE'].unique()) if c != REF_ACC]

# Comprobación: ¿hay categorías en test que no existan en train? (relevante para saber si hace falta un manejo especial)
nuevas_via = set(test['TIPO_VIA'].unique()) - set(train['TIPO_VIA'].unique())
nuevas_acc = set(test['TIPO ACCIDENTE'].unique()) - set(train['TIPO ACCIDENTE'].unique())
print('Categorías de TIPO_VIA en test no vistas en train:', nuevas_via if nuevas_via else 'ninguna')
print('Categorías de TIPO ACCIDENTE en test no vistas en train:', nuevas_acc if nuevas_acc else 'ninguna')

Categorías de TIPO_VIA en test no vistas en train: ninguna
Categorías de TIPO ACCIDENTE en test no vistas en train: ninguna

No hay categorías nuevas en `test`, así que no hace falta una estrategia especial para "categoría desconocida" — pero se deja la función preparada con `.reindex(..., fill_value=0)`, que asignaría ceros en todas las columnas *one-hot* (equivalente a la categoría de referencia) si algún día apareciera una categoría no vista, en vez de romper el pipeline.

In [5]:
def one_hot_fit_train(df, col, cats, prefix):
    dummies = pd.get_dummies(df[col], prefix=prefix)
    return dummies.reindex(columns=[f'{prefix}_{c}' for c in cats], fill_value=0).astype(int)

for df_ in [train, test]:
    ohe_via = one_hot_fit_train(df_, 'TIPO_VIA', cats_tipo_via, 'VIA')
    ohe_acc = one_hot_fit_train(df_, 'TIPO ACCIDENTE', cats_tipo_acc, 'ACC')
    df_[ohe_via.columns.tolist()] = ohe_via
    df_[ohe_acc.columns.tolist()] = ohe_acc

print(f'Columnas one-hot de TIPO_VIA (referencia = {REF_VIA}):', [c for c in ohe_via.columns])
print(f'Columnas one-hot de TIPO ACCIDENTE (referencia = {REF_ACC}):', [c for c in ohe_acc.columns])

Columnas one-hot de TIPO_VIA (referencia = CALLE): ['VIA_AUTOVIA', 'VIA_AVENIDA', 'VIA_CARRETERA', 'VIA_GLORIETA', 'VIA_OTROS', 'VIA_PASEO', 'VIA_PLAZA', 'VIA_RONDA']
Columnas one-hot de TIPO ACCIDENTE (referencia = COLISIÓN DOBLE): ['ACC_ATROPELLO', 'ACC_CAÍDA BICICLETA', 'ACC_CAÍDA CICLOMOTOR', 'ACC_CAÍDA MOTOCICLETA', 'ACC_CAÍDA VEHÍCULO 3 RUEDAS', 'ACC_CAÍDA VIAJERO BUS', 'ACC_CHOQUE CON OBJETO FIJO', 'ACC_COLISIÓN MÚLTIPLE', 'ACC_OTRAS CAUSAS', 'ACC_VUELCO']

## 4. Meteorología y estado del firme: la trampa de variable ficticia que casi pasa desapercibida

`CPFA_*` (condición meteorológica) y `CPSV_*` (estado del firme) llegaron del preprocesado como 6 y 6 columnas binarias independientes respectivamente. Antes de usarlas todas juntas, conviene comprobar algo que no es evidente a simple vista: ¿son en realidad categorías mutuamente excluyentes (es decir, un *one-hot* ya construido) camuflado en columnas sueltas?


In [6]:
CPFA_COLS = ['CPFA Granizo','CPFA Hielo','CPFA Lluvia','CPFA Niebla','CPFA Seco','CPFA Nieve']
CPSV_COLS = ['CPSV Mojada','CPSV Aceite','CPSV Barro','CPSV Grava Suelta','CPSV Hielo','CPSV Seca Y Limpia']

print('Suma de las 6 columnas CPFA por fila (¿sale casi siempre 1?):')
print(train[CPFA_COLS].sum(axis=1).value_counts())
print()
print('Suma de las 6 columnas CPSV por fila:')
print(train[CPSV_COLS].sum(axis=1).value_counts())

Suma de las 6 columnas CPFA por fila (¿sale casi siempre 1?):
1    58044
2       51
Name: count, dtype: int64

Suma de las 6 columnas CPSV por fila:
1    57693
0      288
2      110
3        4
Name: count, dtype: int64

Confirmado: en el 99.9% de los casos, exactamente una columna `CPFA_*` vale 1 (son mutuamente excluyentes) y algo similar ocurre con `CPSV_*`. Es decir, son un *one-hot* ya construido de dos variables categóricas ("condición meteorológica" y "estado del firme"), no seis variables binarias independientes. Usar las 6 columnas de cada grupo a la vez es exactamente la misma trampa de variable ficticia de la sección anterior — de hecho, se ve directamente en la matriz de correlación:


In [7]:
corr_meteo = train[CPFA_COLS + CPSV_COLS].corr().round(2)
corr_meteo

                    CPFA Granizo  CPFA Hielo  CPFA Lluvia  CPFA Niebla  CPFA Seco  CPFA Nieve  CPSV Mojada  CPSV Aceite  CPSV Barro  CPSV Grava Suelta  CPSV Hielo  CPSV Seca Y Limpia
CPFA Granizo                1.00       -0.00         0.01         0.03      -0.03       -0.00         0.02        -0.00       -0.00              -0.00        0.04               -0.02
CPFA Hielo                 -0.00        1.00        -0.01        -0.00      -0.08       -0.00         0.01        -0.00       -0.00              -0.00        0.59               -0.06
CPFA Lluvia                 0.01       -0.01         1.00        -0.00      -0.98        0.00         0.89        -0.02        0.02              -0.01        0.03               -0.85
CPFA Niebla                 0.03       -0.00        -0.00         1.00      -0.12       -0.00         0.09         0.00       -0.00               0.01       -0.00               -0.09
CPFA Seco                  -0.03       -0.08        -0.98        -0.12       1.00    

`CPFA Lluvia` y `CPFA Seco` correlacionan -0.98: son casi opuestos exactos, como cabía esperar. **Se elimina la categoría de referencia de cada grupo** (`CPFA Seco` y `CPSV Seca Y Limpia`, las condiciones "normales" y más frecuentes), conservando solo las condiciones anómalas — su ausencia ya implica "condición normal", exactamente igual que con `TIPO_VIA`/`TIPO ACCIDENTE` en la sección anterior.


In [8]:
CPFA_ANOMALAS = [c for c in CPFA_COLS if c != 'CPFA Seco']
CPSV_ANOMALAS = [c for c in CPSV_COLS if c != 'CPSV Seca Y Limpia']

print('Variables meteorológicas conservadas:', CPFA_ANOMALAS)
print('Variables de firme conservadas:', CPSV_ANOMALAS)

Variables meteorológicas conservadas: ['CPFA Granizo', 'CPFA Hielo', 'CPFA Lluvia', 'CPFA Niebla', 'CPFA Nieve']
Variables de firme conservadas: ['CPSV Mojada', 'CPSV Aceite', 'CPSV Barro', 'CPSV Grava Suelta', 'CPSV Hielo']

## 5. Construcción de las dos matrices de diseño (operativo y referencia)

Con todo lo anterior ya calculado, se definen explícitamente qué columnas entran en cada uno de los dos modelos. Esta tabla es, en sí misma, la justificación de disponibilidad temporal de cada variable — es importante que quede documentada aquí, porque es la pieza central que sostiene la pregunta de investigación del TFM.


In [9]:
COLS_BASE_OPERATIVO = (
    ['ES_CRUCE', 'ES_FINDE', 'HORA_SIN', 'HORA_COS', 'MES_SIN', 'MES_COS']
    + CPFA_ANOMALAS + CPSV_ANOMALAS
    + ['TASA_GRAVEDAD_HIST_DISTRITO', 'PCT_HIST_PEATON_DISTRITO', 'PCT_HIST_MOTO_DISTRITO']
    + ['INCLUYE_MOTO', 'INCLUYE_BICI']   # tipo de vehículo: reportado habitualmente en el aviso inicial
)
COLS_VIA = [f'VIA_{c}' for c in cats_tipo_via]
COLS_ACC = [f'ACC_{c}' for c in cats_tipo_acc]

FEATURES_OPERATIVO = COLS_BASE_OPERATIVO + COLS_VIA + COLS_ACC

# Variables de persona: solo se conocen después de que lleguen los servicios de emergencia
FEATURES_PERSONA_EXTRA = [
    'INCLUYE_PEATON',        # redundante en parte con TIPO ACCIDENTE=ATROPELLO, pero se deja para el modelo de referencia
    'INCLUYE_EDAD_RIESGO',
    'N_PERSONAS_IMPLICADAS',
    'N_VICTIMAS_LOG',
]
FEATURES_REFERENCIA = FEATURES_OPERATIVO + FEATURES_PERSONA_EXTRA

print(f'Nº de features — modelo operativo:   {len(FEATURES_OPERATIVO)}')
print(f'Nº de features — modelo de referencia: {len(FEATURES_REFERENCIA)}')

Nº de features — modelo operativo:   39
Nº de features — modelo de referencia: 43

**Por qué `INCLUYE_PEATON` va solo al modelo de referencia y no al operativo, a pesar de estar correlacionada con `TIPO ACCIDENTE = ATROPELLO`:** en el aviso inicial se suele saber el tipo de accidente (de ahí que `ACC_ATROPELLO` sí esté en el operativo), pero confirmar que hay específicamente un peatón entre los implicados (y no, por ejemplo, un ciclista atropellado por error de clasificación, o más de una persona con roles distintos) es información que en la práctica se termina de confirmar cuando llegan los servicios — de ahí que se trate como variable de "referencia", no operativa, siguiendo el mismo criterio estricto que con el resto de variables de persona.

## 6. Verificación final: sin categorías nuevas, sin nulos, sin colinealidad severa

Antes de dar el dataset por bueno, tres comprobaciones que deben pasar sí o sí.


In [10]:
X_train_op, X_test_op = train[FEATURES_OPERATIVO].copy(), test[FEATURES_OPERATIVO].copy()
X_train_ref, X_test_ref = train[FEATURES_REFERENCIA].copy(), test[FEATURES_REFERENCIA].copy()
y_train, y_test = train['GRAVE'].copy(), test['GRAVE'].copy()

print('--- Nulos ---')
print('Train operativo:', X_train_op.isna().sum().sum(), '| Test operativo:', X_test_op.isna().sum().sum())
print('Train referencia:', X_train_ref.isna().sum().sum(), '| Test referencia:', X_test_ref.isna().sum().sum())

print()
print('--- Dimensiones ---')
print('Operativo:', X_train_op.shape, X_test_op.shape)
print('Referencia:', X_train_ref.shape, X_test_ref.shape)

--- Nulos ---
Train operativo: 0 | Test operativo: 0
Train referencia: 0 | Test referencia: 0

--- Dimensiones ---
Operativo: (58095, 39) (10678, 39)
Referencia: (58095, 43) (10678, 43)

In [11]:
# --- Colinealidad: pares de variables con correlación absoluta > 0.6 ---
corr = X_train_op.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
pares_altos = (
    upper.stack()
    .reset_index()
    .rename(columns={'level_0': 'var_1', 'level_1': 'var_2', 0: 'correlacion'})
    .query('correlacion > 0.6')
    .sort_values('correlacion', ascending=False)
)
pares_altos

                           var_1                     var_2  correlacion
323                  CPFA Lluvia               CPSV Mojada     0.890193
642  TASA_GRAVEDAD_HIST_DISTRITO    PCT_HIST_MOTO_DISTRITO     0.713115
641  TASA_GRAVEDAD_HIST_DISTRITO  PCT_HIST_PEATON_DISTRITO     0.637983

Quedan tres pares por encima de 0.6, y los tres tienen una explicación sustantiva razonable, no un error de codificación:

- **`CPFA Lluvia` / `CPSV Mojada` (0.89):** tiene sentido — cuando llueve, el firme suele estar mojado. Es una correlación real del mundo, no un artefacto.
- **`TASA_GRAVEDAD_HIST_DISTRITO` / `PCT_HIST_PEATON_DISTRITO` y `PCT_HIST_MOTO_DISTRITO` (0.64 / 0.71):** esperable, ya que los tres son agregados calculados sobre el mismo `DISTRITO` — comparten parte de la varianza por construcción, aunque cada uno capture un aspecto distinto (gravedad global vs. composición de peatones vs. composición de motos).

Ninguno llega al nivel de colinealidad *perfecta* (>0.95) visto en la sección 4 antes de corregirlo, así que no hace falta eliminar ninguna variable adicional. Queda anotado como algo a vigilar si en el notebook de modelización se usa un modelo lineal (regresión logística): conviene revisar el *Variance Inflation Factor (VIF)* de estas variables o aplicar regularización L2, que es robusta a este nivel de correlación moderada.


## 7. Guardado de artefactos

Se guardan las cuatro matrices y los dos vectores objetivo por separado, ya listas para el notebook de modelización — sin necesidad de repetir ningún paso de codificación.


In [12]:
import os

os.makedirs('../data/processed/features', exist_ok=True)

X_train_op.to_csv('../data/processed/features/X_train_operativo.csv', index=False)
X_test_op.to_csv('../data/processed/features/X_test_operativo.csv', index=False)
X_train_ref.to_csv('../data/processed/features/X_train_referencia.csv', index=False)
X_test_ref.to_csv('../data/processed/features/X_test_referencia.csv', index=False)
y_train.to_csv('../data/processed/features/y_train.csv', index=False)
y_test.to_csv('../data/processed/features/y_test.csv', index=False)

print('Ficheros guardados en ../data/processed/features/')
print(f'  X_train_operativo:  {X_train_op.shape}')
print(f'  X_train_referencia: {X_train_ref.shape}')

Ficheros guardados en ../data/processed/features/
  X_train_operativo:  (58095, 39)
  X_train_referencia: (58095, 43)

## Resumen de decisiones tomadas en este notebook

| # | Decisión | Por qué |
|---|---|---|
| 1 | Toda codificación se ajusta solo con `train` y se aplica después a `test` | Evitar fuga de información: en producción, el modelo nunca tendría acceso a estadísticos calculados con datos futuros |
| 2 | Agregados históricos suavizados (`m=50`) para `DISTRITO` sobre `GRAVE`, `INCLUYE_PEATON`, `INCLUYE_MOTO` | Recuperan parte de la señal de las variables de persona sin romper la disponibilidad temporal — es información de contexto histórico, no del accidente actual |
| 3 | `HORA` y `MES` codificadas en seno/coseno | Son variables cíclicas; una codificación lineal introduciría un salto artificial entre 23h/0h y diciembre/enero |
| 4 | `TIPO_VIA` y `TIPO ACCIDENTE` con *one-hot*, `DISTRITO` no | Cardinalidad baja y sin agregados históricos alternativos para las dos primeras; `DISTRITO` ya está representado vía los agregados de la decisión 2, un *one-hot* adicional sería redundante |
| 5 | Categoría de referencia = la más frecuente (`CALLE`, `COLISIÓN DOBLE`), no la primera alfabética | Coeficientes interpretables como "frente al caso más común" |
| 6 | Se detecta y corrige que `CPFA_*` / `CPSV_*` son en realidad un *one-hot* ya construido, y se elimina la categoría de referencia de cada grupo | Sin corregirlo, hay colinealidad casi perfecta (r=-0.98) que perjudicaría a cualquier modelo lineal |
| 7 | Variables de persona (`INCLUYE_PEATON`, `INCLUYE_EDAD_RIESGO`, `N_PERSONAS_IMPLICADAS`, `N_VICTIMAS_LOG`) solo en el modelo de **referencia**, nunca en el operativo | Son información posterior al accidente; incluirlas en el modelo operativo lo haría no desplegable en la práctica |
| 8 | Colinealidad moderada (0.6-0.9) que queda, se documenta pero no se corrige | Tiene explicación sustantiva real (lluvia↔firme mojado; agregados del mismo distrito); se traslada como advertencia al notebook de modelización para el caso de usar modelos lineales |

**Siguiente paso:** `04_Modelizacion.ipynb` — entrenar y comparar el modelo operativo y el de referencia (regresión logística con class_weight + Random Forest/Gradient Boosting), evaluar la diferencia de rendimiento entre ambos, e interpretar los factores de riesgo con SHAP para el análisis explicativo.
